In [0]:

# 1. AUTO LOADER - IMPORTS AND CONFIGURATION

# Purpose:
# Define the source, metadata and target locations.


from pyspark.sql import functions as F

# Landing area containing date-wise folders
landing_path = ("/Volumes/severntrent/bronze/raw_data/""autoloader_landing/")

# Schema and checkpoint metadata location
metadata_path = ("/Volumes/severntrent/bronze/raw_data/""autoloader_metadata/")

# Target schema
target_schema = "severntrent.bronze_dev"

# Source names and their target tables
fact_sources = {
    "FactWork": "fact_work_auto",
    "FactShift": "fact_shift_auto",
    "FactShrinkage": "fact_shrinkage_auto"
}

print("Landing path:", landing_path)
print("Target schema:", target_schema)
print("Fact sources:", list(fact_sources.keys()))

Landing path: /Volumes/severntrent/bronze/raw_data/autoloader_landing/
Target schema: severntrent.bronze_dev
Fact sources: ['FactWork', 'FactShift', 'FactShrinkage']


In [0]:

# 2. CREATE REUSABLE AUTO LOADER FUNCTION

# Purpose:
# 1. Read new CSV files from dated landing folders.
# 2. Select files belonging to the requested fact table.
# 3. Add audit columns.
# 4. Return a streaming DataFrame.
#
# This function does NOT write data into Bronze.


def bronze_autoloader(source_name):

    # 1. Define a separate schema location for each source
    source_key = source_name.lower()
    schema_path = metadata_path + source_key + "/schema/"

    # 2. Read matching CSV files using Auto Loader
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", source_name + "_*.csv")
        .load(landing_path)
    )

    # 3. Add source filename
    df = df.withColumn("source_file",
        F.col("_metadata.file_name")
    )

    # 4. Extract load date from filename
    df = df.withColumn(
        "load_date",
        F.to_date(
            F.regexp_extract(
                F.col("source_file"),
                r"(\d{2}_\d{2}_\d{4})",
                1
            ),
            "dd_MM_yyyy"
        )
    )

    # 5. Add ingestion timestamp
    df = df.withColumn("ingestion_ts",F.current_timestamp())

    # 6. Identify FULL or DELTA load
    df = df.withColumn("load_type",F.when(F.col("load_date") == F.lit("2026-04-01").cast("date"),
            "FULL").otherwise("DELTA")
    )

    return df


print("Reusable Auto Loader function created successfully")

Reusable Auto Loader function created successfully


In [0]:

# 3. CREATE REUSABLE BRONZE WRITE FUNCTION

# Purpose:
# 1. Read new files using our Auto Loader function.
# 2. Write incoming records into Bronze Delta tables.
# 3. Use separate checkpoints for each fact table.
# 4. Process available files and then stop.
#
# This function works for all three fact tables.


def write_bronze(source_name):

    # 1. Identify the target table
    table_name = fact_sources[source_name]
    target_table = target_schema + "." + table_name

    # 2. Define a separate checkpoint for each source
    source_key = table_name.replace("_auto", "")
    checkpoint_path = metadata_path + source_key + "/checkpoint/"

    # 3. Read incoming files using Auto Loader
    df = bronze_autoloader(source_name)

    # 4. Write new records into the Bronze Delta table
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    # 5. Wait until all available files are processed
    query.awaitTermination()

    print(source_name, "Auto Loader ingestion completed")


print("Reusable Bronze write function created successfully")

Reusable Bronze write function created successfully


In [0]:

# 4. EXECUTE AUTO LOADER FOR ALL FACT TABLES

# Purpose:
# 1. Process FactWork, FactShift and FactShrinkage.
# 2. Use the same reusable ingestion functions.
# 3. Automatically discover newly landed CSV files.
# 4. Skip previously processed files using checkpoints.


for source_name in fact_sources:

    print("\nProcessing:", source_name)

    write_bronze(source_name)

print("\nAll fact table ingestion completed")


Processing: FactWork
FactWork Auto Loader ingestion completed

Processing: FactShift
FactShift Auto Loader ingestion completed

Processing: FactShrinkage
FactShrinkage Auto Loader ingestion completed

All fact table ingestion completed


In [0]:

# 5. VALIDATE ALL AUTO LOADER BRONZE TABLES

# Purpose:
# 1. Check records stored in each Bronze Delta table.
# 2. Verify FULL and DELTA loads.
# 3. Confirm that previous records are preserved.
#
# This cell only reads data; it does not change tables.


for source_name, table_name in fact_sources.items():

    # Define target Delta table
    target_table = target_schema + "." + table_name

    # Read saved Bronze data
    df = spark.table(target_table)

    print("\nTable:", target_table)
    print("Total records:", df.count())

    # Display daily record counts
    df.groupBy("load_date", "load_type") \
        .count() \
        .orderBy("load_date") \
        .show()


Table: severntrent.bronze_dev.fact_work_auto
Total records: 9830
+----------+---------+-----+
| load_date|load_type|count|
+----------+---------+-----+
|2026-04-01|     FULL| 7010|
|2026-04-02|    DELTA|  650|
|2026-04-03|    DELTA|  570|
|2026-04-04|    DELTA|  750|
|2026-04-05|    DELTA|  850|
+----------+---------+-----+


Table: severntrent.bronze_dev.fact_shift_auto
Total records: 9830
+----------+---------+-----+
| load_date|load_type|count|
+----------+---------+-----+
|2026-04-01|     FULL| 7010|
|2026-04-02|    DELTA|  650|
|2026-04-03|    DELTA|  570|
|2026-04-04|    DELTA|  750|
|2026-04-05|    DELTA|  850|
+----------+---------+-----+


Table: severntrent.bronze_dev.fact_shrinkage_auto
Total records: 9830
+----------+---------+-----+
| load_date|load_type|count|
+----------+---------+-----+
|2026-04-01|     FULL| 7010|
|2026-04-02|    DELTA|  650|
|2026-04-03|    DELTA|  570|
|2026-04-04|    DELTA|  750|
|2026-04-05|    DELTA|  850|
+----------+---------+-----+

